# 🍳 GPT-2 Recipe Pre-training from Scratch

**Build and pre-train a GPT-2 Mini language model (~50M parameters) on a custom recipe dataset**

---

## 📋 Overview

This notebook trains a GPT-2 language model from scratch on a recipe corpus to generate coherent recipe text.

**Pipeline:**
1. Train custom BPE tokenizer (30K vocabulary)
2. Initialize GPT-2 Mini architecture (6 layers, 512 dim, 8 heads)
3. Pre-train with FP16 mixed precision for 10 epochs
4. Generate recipe text from prompts

---

## ✨ IMPORTANT: Use Structured Dataset Format

**To avoid mixed-up recipe output**, use the structured datasets:

| Phase | Dataset File | Format |
|-------|--------------|--------|
| Pre-training | `Dataset/structured_recipes_pretrain.txt` | `**Title:** ... **Ingredients:** (bulleted) **Instructions:** (numbered)` |
| Fine-tuning | `Dataset/structured_recipes_finetune.jsonl` | Alpaca JSONL with structured responses |

The structured format uses explicit field markers (`**Title:**`, `**Ingredients:**`, `**Instructions:**`) that teach the model to generate well-organized recipes.

---

## ⚙️ Setup Instructions (Google Colab Pro)


1. **Enable A100 GPU**: `Runtime` → `Change runtime type` → `A100 GPU`- `outputs/gpt2-recipe-checkpoints/` — Model checkpoints (epoch 1-10)

2. **Upload structured datasets**: Upload `Dataset/structured_recipes_pretrain.txt` and `Dataset/structured_recipes_finetune.jsonl`- `outputs/tokenizer/` — Trained BPE tokenizer (vocab.json, merges.txt)

3. **Update paths** in Section 0.0 and Section 9.1 below

4. **Run all cells**: `Runtime` → `Run all`## 📁 Outputs



**Estimated Time**: ~2 hours for complete training on 8,500 recipes---


---

# SECTION 0: USER INPUTS & HYPERPARAMETERS

> ⚠️ **CONFIGURE THESE BEFORE RUNNING THE NOTEBOOK**

In [ ]:
# ============================================================================
# SECTION 0.0: USER INPUTS (MODIFY THESE BEFORE RUNNING)
# ============================================================================
"""
📁 Path to your recipe dataset file.
   Upload to Colab runtime first, then set the path here.
   
   ✨ USE STRUCTURED FORMAT FOR BEST RESULTS:
   The structured format uses explicit field markers:
   **Title:**, **Ingredients:**, **Instructions:**, **Serving Suggestion:**
   This helps the model learn distinct recipe sections.
"""

# ⚠️ INPUT REQUIRED: Set paths to your structured datasets
RECIPE_FILE_PATH = "Dataset/structured_recipes_pretrain.txt"  # Pre-training data

print("✓ User inputs configured")
print(f"  Recipe file: {RECIPE_FILE_PATH}")

In [ ]:
# ============================================================================
# SECTION 0.1: TOKENIZER HYPERPARAMETERS
# ============================================================================
"""
Configuration for training the Byte Pair Encoding (BPE) tokenizer.
These settings determine vocabulary size and special token handling.

NOTE: vocab_size of 12,000 is optimized for the recipe corpus.
      Analysis showed ~7,768 unique tokens with min_freq=2.
      12,000 provides headroom for subword splits and unseen variations.
"""

TOKENIZER_CONFIG = {
    "vocab_size": 12_000,           # Target vocabulary size for BPE (optimized for recipe corpus)
    "min_frequency": 2,             # Minimum token frequency to include
    "special_tokens": [
        "[PAD]",                    # Padding token (ID: 0)
        "[UNK]",                    # Unknown token (ID: 1)
        "[BOS]",                    # Beginning of sequence (ID: 2)
        "[EOS]",                    # End of sequence (ID: 3)
    ],
}

print("✓ Tokenizer config loaded")
print(f"  Vocabulary size: {TOKENIZER_CONFIG['vocab_size']:,}")
print(f"  Special tokens: {TOKENIZER_CONFIG['special_tokens']}")

In [ ]:
# ============================================================================
# SECTION 0.2: MODEL ARCHITECTURE HYPERPARAMETERS (GPT-2 Small - UPGRADED)
# ============================================================================
"""
GPT-2 Small configuration: ~125M parameters (UPGRADED from Mini ~50M).
Larger model capacity enables better learning of recipe patterns and coherent generation.

CHANGES FROM PREVIOUS (GPT-2 Mini):
- n_layer: 6 → 12 (doubled depth for better pattern learning)
- n_embd: 512 → 768 (larger embedding for richer representations)
- n_head: 8 → 12 (more attention heads for complex dependencies)
- Parameters: ~50M → ~125M (2.5x increase)

NOTE: vocab_size must match TOKENIZER_CONFIG["vocab_size"] (12,000).
"""

MODEL_CONFIG = {
    "vocab_size": 12_000,           # Must match tokenizer vocab_size
    "n_positions": 1024,            # Reduced from 3000 (most recipes < 800 tokens, saves memory)
    "n_embd": 768,                  # Embedding dimension (UPGRADED: 512 → 768)
    "n_layer": 12,                  # Number of transformer layers (UPGRADED: 6 → 12)
    "n_head": 12,                   # Number of attention heads (UPGRADED: 8 → 12)
    "activation_function": "gelu_new",
    "resid_pdrop": 0.1,             # Residual dropout
    "embd_pdrop": 0.1,              # Embedding dropout
    "attn_pdrop": 0.1,              # Attention dropout
}

# Calculate approximate parameter count
approx_params = (
    MODEL_CONFIG["vocab_size"] * MODEL_CONFIG["n_embd"] +  # Token embeddings
    MODEL_CONFIG["n_positions"] * MODEL_CONFIG["n_embd"] +  # Position embeddings
    MODEL_CONFIG["n_layer"] * (
        4 * MODEL_CONFIG["n_embd"] ** 2 +  # Attention (Q, K, V, O)
        8 * MODEL_CONFIG["n_embd"] ** 2    # FFN (up + down projection)
    )
)

print("✓ Model config loaded (GPT-2 Small - UPGRADED)")
print(f"  Layers: {MODEL_CONFIG['n_layer']}, Embedding: {MODEL_CONFIG['n_embd']}, Heads: {MODEL_CONFIG['n_head']}")
print(f"  Max sequence length: {MODEL_CONFIG['n_positions']:,} tokens")
print(f"  Approximate parameters: ~{approx_params / 1e6:.1f}M")

In [ ]:
# ============================================================================
# SECTION 0.3: TRAINING HYPERPARAMETERS (ENHANCED)
# ============================================================================
"""
ENHANCED Training configuration for GPT-2 Small (~125M params).
Optimized for Google Colab Pro A100 GPU (40GB VRAM).

KEY CHANGES FOR BETTER LEARNING:
- num_train_epochs: 7 → 15 (more repetition for pattern learning)
- learning_rate: 5e-5 → 3e-5 (lower LR for stable training)
- warmup_steps: 500 → 1500 (longer warmup prevents early divergence)
- gradient_accumulation_steps: 4 → 8 (larger effective batch = 32)
- max_grad_norm: Added gradient clipping for stability
- lr_scheduler_type: Added cosine annealing for smooth convergence

Effective batch size = per_device_train_batch_size × gradient_accumulation_steps = 32
"""

TRAINING_CONFIG = {
    "num_train_epochs": 15,                    # INCREASED: 7 → 15 epochs for better learning
    "per_device_train_batch_size": 4,          # Batch size per GPU
    "gradient_accumulation_steps": 8,          # INCREASED: Effective batch = 4 * 8 = 32
    "learning_rate": 3e-5,                     # LOWERED: 5e-5 → 3e-5 for stability
    "weight_decay": 0.01,                      # L2 regularization
    "warmup_steps": 1500,                      # INCREASED: 500 → 1500 for stable start
    "max_grad_norm": 1.0,                      # NEW: Gradient clipping for stability
    "lr_scheduler_type": "cosine",             # NEW: Cosine annealing for smooth decay
    "fp16": True,                              # Mixed precision training
    "logging_dir": "./logs",                   # TensorBoard logs directory
    "logging_steps": 50,                       # More frequent logging
    "save_strategy": "epoch",                  # Save checkpoint every epoch
    "save_total_limit": 5,                     # Keep last 5 checkpoints
    "output_dir": "./gpt2-recipe-checkpoints", # Checkpoint directory
    "report_to": "none",                       # Disable wandb/tensorboard
    "dataloader_num_workers": 2,               # Data loading workers
    "seed": 42,                                # Random seed for reproducibility
    "label_smoothing_factor": 0.1,             # NEW: Label smoothing to reduce overfitting
}

print("✓ Training config loaded (ENHANCED for GPT-2 Small)")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']} (increased from 7)")
print(f"  Batch size: {TRAINING_CONFIG['per_device_train_batch_size']} × {TRAINING_CONFIG['gradient_accumulation_steps']} = {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']} effective")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']} (lowered from 5e-5)")
print(f"  Warmup steps: {TRAINING_CONFIG['warmup_steps']} (increased from 500)")
print(f"  Scheduler: {TRAINING_CONFIG['lr_scheduler_type']} (new)")
print(f"  Gradient clipping: {TRAINING_CONFIG['max_grad_norm']} (new)")
print(f"  Label smoothing: {TRAINING_CONFIG['label_smoothing_factor']} (new)")

In [ ]:
# ============================================================================
# SECTION 0.4: INFERENCE HYPERPARAMETERS
# ============================================================================
"""
Generation settings for recipe text inference.
These control the diversity and quality of generated text.
"""

INFERENCE_CONFIG = {
    "max_new_tokens": 200,          # Maximum tokens to generate
    "temperature": 0.8,             # Sampling temperature (higher = more random)
    "top_k": 50,                    # Top-k sampling
    "top_p": 0.92,                  # Nucleus sampling threshold
    "do_sample": True,              # Enable sampling (vs greedy)
    "repetition_penalty": 1.1,      # Penalize repeated tokens
    "no_repeat_ngram_size": 3,      # Prevent repeating n-grams of this size
}

print("✓ Inference config loaded")
print(f"  Max new tokens: {INFERENCE_CONFIG['max_new_tokens']}")
print(f"  Temperature: {INFERENCE_CONFIG['temperature']}")
print(f"  Sampling: top_k={INFERENCE_CONFIG['top_k']}, top_p={INFERENCE_CONFIG['top_p']}")

---

# SECTION 1: ENVIRONMENT SETUP

> Install dependencies and verify GPU availability

In [ ]:
# ============================================================================
# SECTION 1.1: INSTALL DEPENDENCIES
# ============================================================================
"""
Install required packages. PyTorch is pre-installed in Colab.
Using -q for quiet installation to reduce output noise.
"""

!pip install -q transformers tokenizers datasets matplotlib seaborn

print("✓ Dependencies installed")

In [ ]:
# ============================================================================
# SECTION 1.2: IMPORT LIBRARIES
# ============================================================================
"""
Import all required libraries for tokenizer training, model building, and visualization.
"""

import os
import json
from pathlib import Path

# PyTorch (pre-installed in Colab)
import torch
from torch.utils.data import Dataset, DataLoader

# Hugging Face ecosystem
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Visualization (constitution-mandated)
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✓ All libraries imported successfully")
print(f"  PyTorch version: {torch.__version__}")
print(f"  Transformers imported")

In [ ]:
# ============================================================================
# SECTION 1.3: GPU AVAILABILITY CHECK
# ============================================================================
"""
Verify GPU availability and print device information.
This notebook is optimized for A100 GPU (40GB VRAM).
"""

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print("✓ GPU Available!")
    print(f"  Device: {gpu_name}")
    print(f"  VRAM: {gpu_memory:.1f} GB")
    
    # Check if A100
    if "A100" in gpu_name:
        print("  ✓ A100 GPU detected - optimal configuration")
    else:
        print(f"  ⚠️ Warning: Expected A100, got {gpu_name}")
        print("    Training may be slower or require batch size adjustment")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected - training will be very slow")
    print("  Recommendation: Enable GPU in Runtime > Change runtime type > A100")

print(f"\n  Using device: {device}")

---

# SECTION 2: DATA LOADING

> Load and explore the recipe dataset

In [ ]:
# ============================================================================
# SECTION 2.1: LOAD RECIPE TEXT FILE
# ============================================================================
"""
Load the recipe dataset from a plain text file.
Expected format: One recipe per line with [BOS]/[EOS] markers.
"""

import os

# Check if file exists
if not os.path.exists(RECIPE_FILE_PATH):
    raise FileNotFoundError(
        f"Recipe file not found: {RECIPE_FILE_PATH}\n"
        "Please upload your recipe file to the Colab runtime and update RECIPE_FILE_PATH."
    )

# Load recipes
with open(RECIPE_FILE_PATH, "r", encoding="utf-8") as f:
    recipes = [line.strip() for line in f if line.strip()]

print(f"✓ Loaded {len(recipes):,} recipes from {RECIPE_FILE_PATH}")
print("-" * 60)

# Display sample recipe
sample = recipes[0][:800] if len(recipes[0]) > 800 else recipes[0]
# Convert \n markers to actual newlines for display
sample_display = sample.replace(' \\n ', '\n')
print(f"\n📄 Sample recipe (first entry):")
print("-" * 60)
print(sample_display + "..." if len(recipes[0]) > 800 else sample_display)
print("-" * 60)

In [ ]:
# ============================================================================
# SECTION 2.2: DATA EXPLORATION & STATISTICS
# ============================================================================
"""
Compute and display statistics about the recipe dataset.
"""

import numpy as np

# Calculate recipe lengths (in characters)
recipe_lengths = [len(recipe) for recipe in recipes]

print("📊 Dataset Statistics")
print("=" * 40)
print(f"  Total recipes: {len(recipes):,}")
print(f"  Min length: {min(recipe_lengths):,} chars")
print(f"  Max length: {max(recipe_lengths):,} chars")
print(f"  Mean length: {np.mean(recipe_lengths):,.1f} chars")
print(f"  Median length: {np.median(recipe_lengths):,.1f} chars")
print(f"  Std deviation: {np.std(recipe_lengths):,.1f} chars")

# Check for [BOS] and [EOS] markers
bos_count = sum(1 for r in recipes if "[BOS]" in r)
eos_count = sum(1 for r in recipes if "[EOS]" in r)
print(f"\n📌 Special Token Coverage")
print(f"  Recipes with [BOS]: {bos_count:,} ({100*bos_count/len(recipes):.1f}%)")
print(f"  Recipes with [EOS]: {eos_count:,} ({100*eos_count/len(recipes):.1f}%)")

In [ ]:
# ============================================================================
# SECTION 2.3: VISUALIZE RECIPE LENGTH DISTRIBUTION
# ============================================================================
"""
Plot the distribution of recipe lengths using seaborn histogram.
"""

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(recipe_lengths, bins=50, kde=True, ax=ax, color="steelblue")

ax.set_xlabel("Recipe Length (characters)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Distribution of Recipe Lengths in Dataset", fontsize=14)

# Add vertical line for max sequence length (in chars, approximating 3000 tokens)
# Rough estimate: 1 token ≈ 4 characters
approx_max_chars = MODEL_CONFIG["n_positions"] * 4
ax.axvline(x=approx_max_chars, color="red", linestyle="--", linewidth=2, 
           label=f"Max seq length (~{approx_max_chars:,} chars)")
ax.legend()

plt.tight_layout()
plt.show()

# Count recipes that will be truncated
truncated_count = sum(1 for l in recipe_lengths if l > approx_max_chars)
print(f"\n⚠️ Recipes exceeding max length: {truncated_count:,} ({100*truncated_count/len(recipes):.1f}%)")
print("   These will be truncated during tokenization.")

In [ ]:
# ============================================================================
# SECTION 2.4: VALIDATE STRUCTURED FORMAT
# ============================================================================
"""
Verify that the dataset uses the structured format with explicit field markers.
This is critical for ensuring the model learns distinct section boundaries.

Required markers:
- **Title:** - Recipe name
- **Ingredients:** - Bulleted list with '-'
- **Instructions:** - Numbered list with '1.', '2.', etc.
- **Serving Suggestion:** - Optional serving guidance
"""

def validate_structured_format(recipes, sample_size=100):
    """
    Validate that recipes contain structured format markers.
    
    Args:
        recipes: List of recipe strings
        sample_size: Number of recipes to sample for detailed validation
        
    Returns:
        dict: Validation results with counts and examples
    """
    results = {
        "total": len(recipes),
        "has_title": 0,
        "has_ingredients": 0,
        "has_instructions": 0,
        "has_serving": 0,
        "has_bulleted_list": 0,
        "has_numbered_steps": 0,
        "fully_structured": 0,
        "issues": []
    }
    
    # Check all recipes for required markers
    for i, recipe in enumerate(recipes):
        has_title = "**Title:**" in recipe
        has_ingredients = "**Ingredients:**" in recipe
        has_instructions = "**Instructions:**" in recipe
        has_serving = "**Serving Suggestion:**" in recipe
        has_bullets = " - " in recipe or "\\n- " in recipe
        has_numbers = any(f"{n}." in recipe for n in range(1, 20))
        
        if has_title:
            results["has_title"] += 1
        if has_ingredients:
            results["has_ingredients"] += 1
        if has_instructions:
            results["has_instructions"] += 1
        if has_serving:
            results["has_serving"] += 1
        if has_bullets:
            results["has_bulleted_list"] += 1
        if has_numbers:
            results["has_numbered_steps"] += 1
            
        # Fully structured = has all three required markers + proper formatting
        if has_title and has_ingredients and has_instructions and has_bullets and has_numbers:
            results["fully_structured"] += 1
        elif i < sample_size:  # Track issues for sample
            issues = []
            if not has_title:
                issues.append("missing **Title:**")
            if not has_ingredients:
                issues.append("missing **Ingredients:**")
            if not has_instructions:
                issues.append("missing **Instructions:**")
            if not has_bullets:
                issues.append("missing bulleted list (-)")
            if not has_numbers:
                issues.append("missing numbered steps")
            if issues:
                results["issues"].append((i, issues))
    
    return results

# Run validation
print("🔍 Validating Structured Format")
print("=" * 60)

validation = validate_structured_format(recipes)

# Calculate percentages
pct = lambda x: 100 * validation[x] / validation["total"]

print(f"\n📋 Field Marker Coverage (n={validation['total']:,}):")
print(f"  ✓ **Title:**              {validation['has_title']:,} ({pct('has_title'):.1f}%)")
print(f"  ✓ **Ingredients:**        {validation['has_ingredients']:,} ({pct('has_ingredients'):.1f}%)")
print(f"  ✓ **Instructions:**       {validation['has_instructions']:,} ({pct('has_instructions'):.1f}%)")
print(f"  ○ **Serving Suggestion:** {validation['has_serving']:,} ({pct('has_serving'):.1f}%)")

print(f"\n📝 Formatting Coverage:")
print(f"  ✓ Bulleted ingredients (-): {validation['has_bulleted_list']:,} ({pct('has_bulleted_list'):.1f}%)")
print(f"  ✓ Numbered steps (1., 2.):  {validation['has_numbered_steps']:,} ({pct('has_numbered_steps'):.1f}%)")

print(f"\n🎯 Fully Structured Recipes: {validation['fully_structured']:,} ({pct('fully_structured'):.1f}%)")

# Quality gate
QUALITY_THRESHOLD = 90.0
is_quality_pass = pct('fully_structured') >= QUALITY_THRESHOLD

if is_quality_pass:
    print(f"\n✅ QUALITY GATE PASSED: {pct('fully_structured'):.1f}% >= {QUALITY_THRESHOLD}% threshold")
    print("   Dataset is ready for training with structured format!")
else:
    print(f"\n⚠️ QUALITY WARNING: {pct('fully_structured'):.1f}% < {QUALITY_THRESHOLD}% threshold")
    print("   Consider using the structured dataset: Dataset/structured_recipes_pretrain.txt")
    
    # Show sample issues
    if validation["issues"]:
        print(f"\n📍 Sample Issues (first 5):")
        for idx, issues in validation["issues"][:5]:
            print(f"   Recipe {idx}: {', '.join(issues)}")

# Show a well-formatted example
print("\n" + "=" * 60)
print("📄 Sample Structured Recipe (for verification):")
print("-" * 60)

# Find first fully structured recipe
for recipe in recipes[:10]:
    if all(marker in recipe for marker in ["**Title:**", "**Ingredients:**", "**Instructions:**"]):
        # Pretty print with actual newlines
        display_recipe = recipe.replace(" \\n ", "\n").replace("\\n", "\n")
        # Truncate for display
        if len(display_recipe) > 1000:
            display_recipe = display_recipe[:1000] + "\n... [truncated]"
        print(display_recipe)
        break
else:
    print("⚠️ No fully structured recipes found in first 10 entries!")
    print("   First recipe preview:")
    print(recipes[0][:500] if recipes else "No recipes loaded")

---

# SECTION 3: TOKENIZER TRAINING

> Train a custom Byte Pair Encoding (BPE) tokenizer on the recipe corpus

In [ ]:
# ============================================================================
# SECTION 3.1: TRAIN BPE TOKENIZER
# ============================================================================
# Train a Byte Pair Encoding tokenizer from scratch on recipe corpus

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors

# Initialize BPE tokenizer
tokenizer = Tokenizer(models.BPE(unk_token=TOKENIZER_CONFIG["special_tokens"][1]))

# Set pre-tokenizer (whitespace-based splitting)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Configure BPE trainer
trainer = trainers.BpeTrainer(
    vocab_size=TOKENIZER_CONFIG["vocab_size"],
    min_frequency=TOKENIZER_CONFIG["min_frequency"],
    special_tokens=TOKENIZER_CONFIG["special_tokens"],
    show_progress=True
)

# Train on recipe corpus
print("Training BPE tokenizer on recipe corpus...")
tokenizer.train_from_iterator(recipes, trainer=trainer)

# Add post-processor for BOS/EOS tokens
bos_token = TOKENIZER_CONFIG["special_tokens"][2]  # [BOS]
eos_token = TOKENIZER_CONFIG["special_tokens"][3]  # [EOS]
bos_id = tokenizer.token_to_id(bos_token)
eos_id = tokenizer.token_to_id(eos_token)

tokenizer.post_processor = processors.TemplateProcessing(
    single=f"{bos_token}:0 $A:0 {eos_token}:0",
    special_tokens=[
        (bos_token, bos_id),
        (eos_token, eos_id),
    ],
)

print(f"✓ Tokenizer trained successfully!")
print(f"  Vocabulary size: {tokenizer.get_vocab_size():,}")

In [ ]:
# ============================================================================
# SECTION 3.2: SAVE TOKENIZER TO DISK
# ============================================================================
# Persist trained tokenizer for checkpoint recovery

import os

TOKENIZER_PATH = "recipe_tokenizer"
os.makedirs(TOKENIZER_PATH, exist_ok=True)

# Save the raw tokenizer
tokenizer.save(os.path.join(TOKENIZER_PATH, "tokenizer.json"))

print(f"✓ Tokenizer saved to: {TOKENIZER_PATH}/")

In [ ]:
# ============================================================================
# SECTION 3.3: WRAP IN GPT2TokenizerFast
# ============================================================================
# Create HuggingFace-compatible tokenizer wrapper for Trainer API

from transformers import GPT2TokenizerFast

# Load trained tokenizer into HuggingFace wrapper
hf_tokenizer = GPT2TokenizerFast(
    tokenizer_file=os.path.join(TOKENIZER_PATH, "tokenizer.json"),
    bos_token=TOKENIZER_CONFIG["special_tokens"][2],
    eos_token=TOKENIZER_CONFIG["special_tokens"][3],
    unk_token=TOKENIZER_CONFIG["special_tokens"][1],
    pad_token=TOKENIZER_CONFIG["special_tokens"][0],
)

# Set padding side for causal LM (left padding)
hf_tokenizer.padding_side = "left"

print(f"✓ GPT2TokenizerFast wrapper created!")
print(f"  PAD token: {hf_tokenizer.pad_token} (id: {hf_tokenizer.pad_token_id})")
print(f"  UNK token: {hf_tokenizer.unk_token} (id: {hf_tokenizer.unk_token_id})")
print(f"  BOS token: {hf_tokenizer.bos_token} (id: {hf_tokenizer.bos_token_id})")
print(f"  EOS token: {hf_tokenizer.eos_token} (id: {hf_tokenizer.eos_token_id})")

In [ ]:
# ============================================================================
# SECTION 3.4: TOKENIZER VALIDATION DEMO
# ============================================================================
# Demonstrate tokenization on sample recipes

print("=" * 60)
print("TOKENIZER VALIDATION DEMO")
print("=" * 60)

# Sample recipes for demo
sample_recipes = recipes[:3]

for i, recipe in enumerate(sample_recipes):
    print(f"\n--- Recipe {i+1} ---")
    print(f"Original (first 100 chars): {recipe[:100]}...")
    
    # Tokenize
    encoded = hf_tokenizer(recipe, truncation=True, max_length=MODEL_CONFIG["n_positions"])
    tokens = hf_tokenizer.convert_ids_to_tokens(encoded["input_ids"][:20])
    
    print(f"Token count: {len(encoded['input_ids'])}")
    print(f"First 20 tokens: {tokens}")

print("\n" + "=" * 60)
print("✓ Tokenizer validation complete!")

---

# SECTION 4: DATASET PREPARATION

> Create PyTorch Dataset and DataCollator for training

In [ ]:
# ============================================================================
# SECTION 4.1: RECIPE DATASET CLASS
# ============================================================================
# Custom PyTorch Dataset for tokenized recipes

from torch.utils.data import Dataset

class RecipeDataset(Dataset):
    """PyTorch Dataset for recipe text generation training."""
    
    def __init__(self, recipes: list, tokenizer, max_length: int):
        """
        Args:
            recipes: List of recipe strings
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length
        """
        self.recipes = recipes
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.recipes)
    
    def __getitem__(self, idx):
        recipe = self.recipes[idx]
        
        # Tokenize with truncation
        encoding = self.tokenizer(
            recipe,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"]
        }

print("✓ RecipeDataset class defined!")

In [ ]:
# ============================================================================
# SECTION 4.2: TOKENIZE ALL RECIPES
# ============================================================================
# Create training dataset from recipe corpus

# Create dataset
train_dataset = RecipeDataset(
    recipes=recipes,
    tokenizer=hf_tokenizer,
    max_length=MODEL_CONFIG["n_positions"]
)

print(f"✓ Training dataset created!")
print(f"  Total samples: {len(train_dataset):,}")
print(f"  Max sequence length: {MODEL_CONFIG['n_positions']:,}")

# Sample verification
sample = train_dataset[0]
print(f"\n  Sample 0 token count: {len(sample['input_ids'])}")

In [ ]:
# ============================================================================
# SECTION 4.3: DATA COLLATOR FOR CAUSAL LM
# ============================================================================
# Configure DataCollator for language modeling (shifts labels automatically)

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=False,  # Causal LM (not masked LM)
)

print("✓ DataCollatorForLanguageModeling configured!")
print("  Mode: Causal Language Modeling (mlm=False)")
print("  Labels are automatically shifted for next-token prediction")

---

# SECTION 5: MODEL INITIALIZATION

> Configure and instantiate GPT-2 Mini architecture

In [ ]:
# ============================================================================
# SECTION 5.1: CREATE GPT2CONFIG
# ============================================================================
# Configure GPT-2 Mini architecture

from transformers import GPT2Config

config = GPT2Config(
    vocab_size=MODEL_CONFIG["vocab_size"],
    n_positions=MODEL_CONFIG["n_positions"],
    n_embd=MODEL_CONFIG["n_embd"],
    n_layer=MODEL_CONFIG["n_layer"],
    n_head=MODEL_CONFIG["n_head"],
    # n_inner defaults to 4 * n_embd = 2048
    activation_function=MODEL_CONFIG["activation_function"],
    resid_pdrop=MODEL_CONFIG["resid_pdrop"],
    embd_pdrop=MODEL_CONFIG["embd_pdrop"],
    attn_pdrop=MODEL_CONFIG["attn_pdrop"],
    bos_token_id=hf_tokenizer.bos_token_id,
    eos_token_id=hf_tokenizer.eos_token_id,
    pad_token_id=hf_tokenizer.pad_token_id,
)

print("✓ GPT2Config created!")
print(f"  Architecture: GPT-2 Mini")
print(f"  Vocab size: {config.vocab_size:,}")
print(f"  Max positions: {config.n_positions:,}")
print(f"  Embedding dim: {config.n_embd}")
print(f"  Layers: {config.n_layer}")
print(f"  Heads: {config.n_head}")
print(f"  FFN inner dim: {config.n_inner} (default: 4 * n_embd)")

In [ ]:
# ============================================================================
# SECTION 5.2: INSTANTIATE GPT2LMHeadModel
# ============================================================================
# Create model from scratch (random initialization)

from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel(config)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✓ GPT2LMHeadModel instantiated!")
print(f"  Device: {device}")

In [ ]:
# ============================================================================
# SECTION 5.3: MODEL PARAMETER COUNT
# ============================================================================
# Display total trainable parameters

def count_parameters(model):
    """Count trainable and total parameters."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable_params, total_params = count_parameters(model)

print("=" * 60)
print("MODEL PARAMETER SUMMARY")
print("=" * 60)
print(f"  Total parameters:     {total_params:>15,}")
print(f"  Trainable parameters: {trainable_params:>15,}")
print(f"  Approximate size:     {total_params * 4 / 1e6:>12.2f} MB (FP32)")
print(f"  Approximate size:     {total_params * 2 / 1e6:>12.2f} MB (FP16)")
print("=" * 60)

---

# SECTION 6: TRAINING

> Configure TrainingArguments, initialize Trainer, and execute training loop

In [ ]:
# ============================================================================
# SECTION 6.1: TRAINING ARGUMENTS
# ============================================================================
# Configure HuggingFace Trainer hyperparameters

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=TRAINING_CONFIG["output_dir"],
    overwrite_output_dir=True,
    
    # Training duration
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    
    # Batch size and accumulation
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    # Optimizer settings
    learning_rate=TRAINING_CONFIG["learning_rate"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    warmup_steps=TRAINING_CONFIG["warmup_steps"],
    
    # Mixed precision
    fp16=TRAINING_CONFIG["fp16"],
    
    # Logging
    logging_dir=TRAINING_CONFIG["logging_dir"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    
    # Checkpointing
    save_strategy=TRAINING_CONFIG["save_strategy"],
    save_total_limit=TRAINING_CONFIG["save_total_limit"],
    
    # Misc
    dataloader_num_workers=TRAINING_CONFIG["dataloader_num_workers"],
    seed=TRAINING_CONFIG["seed"],
    report_to=TRAINING_CONFIG["report_to"],
)

print("✓ TrainingArguments configured!")
print(f"  Output directory: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
# ============================================================================
# SECTION 6.2: INITIALIZE TRAINER
# ============================================================================
# Create HuggingFace Trainer instance

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=hf_tokenizer,
)

print("✓ Trainer initialized!")
print(f"  Model: {model.__class__.__name__}")
print(f"  Dataset size: {len(train_dataset):,} samples")

In [ ]:
# ============================================================================
# SECTION 6.3: EXECUTE TRAINING
# ============================================================================
# Run the training loop

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"  Model: GPT-2 Mini ({total_params:,} parameters)")
print(f"  Dataset: {len(train_dataset):,} recipes")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  Device: {device}")
print("=" * 60)

# Train the model
train_result = trainer.train()

# Print training summary
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Total steps: {train_result.global_step:,}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print("=" * 60)

In [ ]:
# ============================================================================
# SECTION 6.4: PLOT TRAINING LOSS CURVE
# ============================================================================
# Visualize training progress

# Extract loss history from trainer
loss_history = [log["loss"] for log in trainer.state.log_history if "loss" in log]
steps = [log["step"] for log in trainer.state.log_history if "loss" in log]

# Create loss curve plot
plt.figure(figsize=(12, 6))
plt.plot(steps, loss_history, 'b-', linewidth=2, alpha=0.8)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("GPT-2 Recipe Pre-training Loss Curve", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3)

# Add epoch markers
samples_per_epoch = len(train_dataset)
steps_per_epoch = samples_per_epoch // (TRAINING_CONFIG["per_device_train_batch_size"] * TRAINING_CONFIG["gradient_accumulation_steps"])
for epoch in range(1, TRAINING_CONFIG["num_train_epochs"] + 1):
    epoch_step = epoch * steps_per_epoch
    if epoch_step <= max(steps):
        plt.axvline(x=epoch_step, color='r', linestyle='--', alpha=0.5, label=f'Epoch {epoch}' if epoch == 1 else '')

plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig("training_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Training loss curve saved to: training_loss_curve.png")

---

# SECTION 7: INFERENCE

> Generate recipe text using the trained model

In [ ]:
# ============================================================================
# SECTION 7.1: LOAD BEST CHECKPOINT
# ============================================================================
# Load the best checkpoint for inference

import glob

# Find the latest checkpoint
checkpoint_dirs = glob.glob(os.path.join(TRAINING_CONFIG["output_dir"], "checkpoint-*"))
if checkpoint_dirs:
    latest_checkpoint = max(checkpoint_dirs, key=os.path.getctime)
    print(f"✓ Loading checkpoint: {latest_checkpoint}")
    model = GPT2LMHeadModel.from_pretrained(latest_checkpoint)
    model = model.to(device)
else:
    print("ℹ Using current model (no checkpoint found)")

model.eval()
print(f"✓ Model ready for inference on {device}")

In [ ]:
# ============================================================================
# SECTION 7.2: RECIPE GENERATION FUNCTION
# ============================================================================
# Define text generation function with configurable parameters

def generate_recipe(prompt: str, max_new_tokens: int = None, temperature: float = None,
                   top_k: int = None, top_p: float = None, do_sample: bool = True) -> str:
    """
    Generate recipe text from a prompt.
    
    Args:
        prompt: Starting text for generation
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (higher = more creative)
        top_k: Top-k sampling parameter
        top_p: Nucleus sampling parameter
        do_sample: Whether to use sampling (False = greedy)
    
    Returns:
        Generated recipe text
    """
    # Use defaults from config if not specified
    max_new_tokens = max_new_tokens or INFERENCE_CONFIG["max_new_tokens"]
    temperature = temperature or INFERENCE_CONFIG["temperature"]
    top_k = top_k or INFERENCE_CONFIG["top_k"]
    top_p = top_p or INFERENCE_CONFIG["top_p"]
    
    # Tokenize prompt
    inputs = hf_tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=hf_tokenizer.pad_token_id,
            eos_token_id=hf_tokenizer.eos_token_id,
            repetition_penalty=INFERENCE_CONFIG["repetition_penalty"],
            no_repeat_ngram_size=INFERENCE_CONFIG["no_repeat_ngram_size"],
        )
    
    # Decode and return
    generated_text = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

print("✓ generate_recipe() function defined!")

In [ ]:
# ============================================================================
# SECTION 7.3: INTERACTIVE GENERATION EXAMPLES
# ============================================================================
# Demonstrate recipe generation with various prompts

print("=" * 60)
print("RECIPE GENERATION EXAMPLES")
print("=" * 60)

# Example prompts
prompts = [
    "Chocolate Cake:",
    "Pasta with",
    "Grilled Chicken",
    "Easy breakfast",
]

for prompt in prompts:
    print(f"\n{'─' * 60}")
    print(f"PROMPT: {prompt}")
    print('─' * 60)
    
    generated = generate_recipe(prompt)
    print(generated[:500] + "..." if len(generated) > 500 else generated)

print("\n" + "=" * 60)
print("✓ Generation examples complete!")
print("=" * 60)

---

# SECTION 8: EXPORT & CLEANUP

> Save final model artifacts and prepare for download

In [ ]:
# ============================================================================
# SECTION 8.1: SAVE FINAL MODEL
# ============================================================================
# Save model and tokenizer for future use

FINAL_MODEL_PATH = "gpt2_recipe_final"

# Save model
model.save_pretrained(FINAL_MODEL_PATH)
print(f"✓ Model saved to: {FINAL_MODEL_PATH}/")

# Save tokenizer
hf_tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"✓ Tokenizer saved to: {FINAL_MODEL_PATH}/")

# List saved files
saved_files = os.listdir(FINAL_MODEL_PATH)
print(f"\nSaved artifacts:")
for f in saved_files:
    size = os.path.getsize(os.path.join(FINAL_MODEL_PATH, f)) / 1e6
    print(f"  {f}: {size:.2f} MB")

In [ ]:
# ============================================================================
# SECTION 8.2: DOWNLOAD ARTIFACTS (COLAB)
# ============================================================================
# Zip and download model artifacts for local storage

import shutil

# Create zip archive
zip_name = "gpt2_recipe_model"
shutil.make_archive(zip_name, 'zip', FINAL_MODEL_PATH)
print(f"✓ Created archive: {zip_name}.zip")

# Download in Colab (uncomment when running in Colab)
# from google.colab import files
# files.download(f"{zip_name}.zip")

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)
print(f"  Model: GPT-2 Mini (~{total_params:,} parameters)")
print(f"  Dataset: {len(train_dataset):,} recipes")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  Final Loss: {train_result.training_loss:.4f}")
print("=" * 60)
print("\nTo download in Colab, uncomment the files.download() line above.")
print("To load the model later:")
print(f"  model = GPT2LMHeadModel.from_pretrained('{FINAL_MODEL_PATH}')")
print(f"  tokenizer = GPT2TokenizerFast.from_pretrained('{FINAL_MODEL_PATH}')")

---

# ═══════════════════════════════════════════════════════════════════════════
# PHASE 2: INSTRUCTION FINE-TUNING (ALIGNMENT)
# ═══════════════════════════════════════════════════════════════════════════

> **Goal**: Fine-tune the pre-trained model to follow user instructions in Alpaca-style format

## 📋 Phase 2 Overview

After pre-training on raw recipe text (Phase 1), we now align the model to follow instructions:

1. **Load Phase 1 Artifacts**: Model and tokenizer from pre-training
2. **Prepare Instruction Dataset**: Alpaca-style JSONL with `instruction` and `response`
3. **Format Training Data**: `### Instruction: {instruction}\n\n### Response: {response}[EOS]`
4. **Fine-tune with Lower LR**: Preserve pre-trained knowledge while learning instruction-following
5. **Interactive Inference**: User inputs a request, model generates a recipe

---

# SECTION 9: PHASE 2 CONFIGURATION

> Configure instruction fine-tuning hyperparameters and file paths

In [ ]:
# ============================================================================
# SECTION 9.1: PHASE 2 USER INPUTS
# ============================================================================
"""
📁 Path to your instruction dataset file (Alpaca-style JSONL).
   Format: {"instruction": "...", "response": "..."}
   One JSON object per line.
"""

import os

# ⚠️ INPUT REQUIRED: Set paths for Phase 2
INSTRUCTION_FILE_PATH = "Dataset/structured_recipes_finetune.jsonl"

# Path to Phase 1 pre-trained model (from Section 8.1)
PRETRAINED_MODEL_PATH = "gpt2_recipe_final"

print("✓ Phase 2 input paths configured")
print(f"  Instruction dataset: {INSTRUCTION_FILE_PATH}")
print(f"  Pre-trained model: {PRETRAINED_MODEL_PATH}")

In [ ]:
# ============================================================================
# SECTION 9.2: FINE-TUNING HYPERPARAMETERS (ENHANCED)
# ============================================================================
"""
ENHANCED Fine-tuning configuration for GPT-2 Small (~125M params).
Uses even lower learning rates and more epochs for instruction alignment.

KEY CHANGES:
- num_train_epochs: 3 → 8 (more epochs for instruction learning)
- learning_rate: 1e-5 → 5e-6 (very low LR to preserve pre-trained knowledge)
- warmup_ratio: 0.1 → 0.15 (longer warmup for fine-tuning stability)
- max_grad_norm: Added gradient clipping
- lr_scheduler_type: Cosine annealing
"""

FINETUNE_CONFIG = {
    "num_train_epochs": 8,                      # INCREASED: 3 → 8 epochs
    "per_device_train_batch_size": 2,           # Smaller batch for instruction data
    "gradient_accumulation_steps": 16,          # INCREASED: Effective batch = 2 * 16 = 32
    "learning_rate": 5e-6,                      # LOWERED: 1e-5 → 5e-6 (preserve knowledge)
    "weight_decay": 0.01,                       # L2 regularization
    "warmup_ratio": 0.15,                       # INCREASED: 15% warmup
    "max_grad_norm": 1.0,                       # NEW: Gradient clipping
    "lr_scheduler_type": "cosine",              # NEW: Cosine annealing
    "fp16": True,                               # Mixed precision training
    "logging_dir": "./logs_finetune",           # Separate logs for fine-tuning
    "logging_steps": 50,                        # More frequent logging
    "save_strategy": "epoch",                   # Save checkpoint every epoch
    "save_total_limit": 3,                      # Keep last 3 checkpoints
    "output_dir": "./gpt2-recipe-instruct",     # Fine-tuned model checkpoints
    "report_to": "none",                        # Disable wandb/tensorboard
    "dataloader_num_workers": 2,                # Data loading workers
    "seed": 42,                                 # Random seed
    "max_seq_length": 1024,                     # Match model n_positions
    "label_smoothing_factor": 0.1,              # NEW: Reduce overfitting
}

print("✓ Fine-tuning config loaded (ENHANCED)")
print(f"  Epochs: {FINETUNE_CONFIG['num_train_epochs']} (increased from 3)")
print(f"  Batch size: {FINETUNE_CONFIG['per_device_train_batch_size']} × {FINETUNE_CONFIG['gradient_accumulation_steps']} = {FINETUNE_CONFIG['per_device_train_batch_size'] * FINETUNE_CONFIG['gradient_accumulation_steps']} effective")
print(f"  Learning rate: {FINETUNE_CONFIG['learning_rate']} (very low to preserve knowledge)")
print(f"  Warmup ratio: {FINETUNE_CONFIG['warmup_ratio']} (15%)")
print(f"  Scheduler: {FINETUNE_CONFIG['lr_scheduler_type']}")

---

# SECTION 10: LOAD PHASE 1 ARTIFACTS

> Load pre-trained model and tokenizer from Phase 1

In [ ]:
# ============================================================================
# SECTION 10.1: LOAD PRE-TRAINED MODEL & TOKENIZER
# ============================================================================
"""
Load the model and tokenizer from Phase 1 pre-training.
These artifacts contain domain knowledge learned from the recipe corpus.
"""

import os
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# Get the content path for Colab compatibility
try:
    from google.colab import drive
    CONTENT_PATH = "/content"
except ImportError:
    CONTENT_PATH = "."

# Construct full path to pre-trained model
PRETRAINED_MODEL_FULL_PATH = os.path.join(CONTENT_PATH, PRETRAINED_MODEL_PATH)

# Verify Phase 1 artifacts exist
if not os.path.exists(PRETRAINED_MODEL_FULL_PATH):
    raise FileNotFoundError(
        f"Pre-trained model not found: {PRETRAINED_MODEL_FULL_PATH}\n"
        "Please complete Phase 1 (pre-training) first, or update PRETRAINED_MODEL_PATH."
    )

# Load tokenizer
ft_tokenizer = GPT2TokenizerFast.from_pretrained(PRETRAINED_MODEL_FULL_PATH)
print(f"✓ Tokenizer loaded from: {PRETRAINED_MODEL_FULL_PATH}")
print(f"  Vocabulary size: {ft_tokenizer.vocab_size:,}")

# Load model
ft_model = GPT2LMHeadModel.from_pretrained(PRETRAINED_MODEL_FULL_PATH)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ft_model = ft_model.to(device)

print(f"✓ Model loaded from: {PRETRAINED_MODEL_FULL_PATH}")
print(f"  Device: {device}")
print(f"  Parameters: {sum(p.numel() for p in ft_model.parameters()):,}")

---

# SECTION 11: INSTRUCTION DATASET PREPARATION

> Load, validate, and format Alpaca-style instruction data

In [ ]:
# ============================================================================
# SECTION 11.1: DATASET VALIDATION FUNCTION
# ============================================================================
"""
Validate Alpaca-style JSONL dataset to ensure required fields exist.
This prevents training failures due to malformed data.
"""

import json
from typing import List, Dict, Tuple

def validate_instruction_dataset(file_path: str) -> Tuple[List[Dict], List[str]]:
    """
    Load and validate an Alpaca-style JSONL instruction dataset.
    
    Args:
        file_path: Path to the JSONL file
        
    Returns:
        Tuple of (valid_samples, error_messages)
        
    Expected format per line:
        {"instruction": "Give me a recipe for...", "response": "Here is a recipe..."}
    """
    valid_samples = []
    errors = []
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Instruction dataset not found: {file_path}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
                
            try:
                sample = json.loads(line)
                
                # Validate required fields
                if "instruction" not in sample:
                    errors.append(f"Line {line_num}: Missing 'instruction' field")
                    continue
                if "response" not in sample:
                    errors.append(f"Line {line_num}: Missing 'response' field")
                    continue
                    
                # Validate non-empty content
                if not sample["instruction"].strip():
                    errors.append(f"Line {line_num}: Empty 'instruction' field")
                    continue
                if not sample["response"].strip():
                    errors.append(f"Line {line_num}: Empty 'response' field")
                    continue
                
                valid_samples.append(sample)
                
            except json.JSONDecodeError as e:
                errors.append(f"Line {line_num}: Invalid JSON - {str(e)}")
    
    return valid_samples, errors

print("✓ validate_instruction_dataset() function defined!")

In [ ]:
# ============================================================================
# SECTION 11.2: LOAD AND VALIDATE INSTRUCTION DATASET
# ============================================================================
"""
Load the instruction dataset and report validation results.
Training will only proceed if all samples are valid.
"""

print("Loading and validating instruction dataset...")
print(f"  File: {INSTRUCTION_FILE_PATH}")
print("-" * 60)

# Validate dataset
instruction_data, validation_errors = validate_instruction_dataset(INSTRUCTION_FILE_PATH)

# Report results
if validation_errors:
    print(f"⚠️ Found {len(validation_errors)} validation errors:")
    for err in validation_errors[:10]:  # Show first 10 errors
        print(f"   {err}")
    if len(validation_errors) > 10:
        print(f"   ... and {len(validation_errors) - 10} more errors")
    print()

print(f"✓ Validation complete!")
print(f"  Valid samples: {len(instruction_data):,}")
print(f"  Invalid samples: {len(validation_errors):,}")

if len(instruction_data) == 0:
    raise ValueError("No valid instruction samples found. Please fix the dataset.")

# Preview samples
print("\n📄 Sample instruction-response pairs:")
print("-" * 60)
for i, sample in enumerate(instruction_data[:2]):
    print(f"\n--- Sample {i+1} ---")
    print(f"Instruction: {sample['instruction'][:100]}...")
    print(f"Response: {sample['response'][:150]}...")
print("-" * 60)

In [ ]:
# ============================================================================
# SECTION 11.3: INSTRUCTION FORMATTING FUNCTION
# ============================================================================
"""
Format instruction-response pairs into the Alpaca template for training.
Template: ### Instruction: {instruction}\n\n### Response: {response}[EOS]
"""

def format_instruction(sample: Dict[str, str], tokenizer) -> str:
    """
    Format a single instruction-response pair into training format.
    
    Args:
        sample: Dictionary with 'instruction' and 'response' keys
        tokenizer: Tokenizer to get EOS token
        
    Returns:
        Formatted string ready for tokenization
    """
    instruction = sample["instruction"].strip()
    response = sample["response"].strip()
    eos_token = tokenizer.eos_token
    
    formatted = f"### Instruction:\n{instruction}\n\n### Response:\n{response}{eos_token}"
    return formatted

# Demonstrate formatting
print("✓ format_instruction() function defined!")
print("\n📝 Format template:")
print("   ### Instruction:")
print("   {user's request}")
print("   ")
print("   ### Response:")
print("   {recipe content}[EOS]")

# Show example
sample_formatted = format_instruction(instruction_data[0], ft_tokenizer)
print(f"\n📄 Example formatted sample (first 300 chars):")
print("-" * 60)
print(sample_formatted[:300] + "...")
print("-" * 60)

In [ ]:
# ============================================================================
# SECTION 11.4: INSTRUCTION DATASET CLASS
# ============================================================================
"""
PyTorch Dataset for instruction fine-tuning.
Formats and tokenizes instruction-response pairs on-the-fly.
"""

from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    """PyTorch Dataset for instruction-tuning with Alpaca-style data."""
    
    def __init__(self, samples: List[Dict], tokenizer, max_length: int):
        """
        Args:
            samples: List of dicts with 'instruction' and 'response' keys
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length
        """
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Format into instruction template
        formatted_text = format_instruction(sample, self.tokenizer)
        
        # Tokenize
        encoding = self.tokenizer(
            formatted_text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"]
        }

print("✓ InstructionDataset class defined!")

In [ ]:
# ============================================================================
# SECTION 11.5: CREATE INSTRUCTION DATASET
# ============================================================================
"""
Instantiate the instruction dataset for fine-tuning.
"""

# Create dataset
instruct_dataset = InstructionDataset(
    samples=instruction_data,
    tokenizer=ft_tokenizer,
    max_length=FINETUNE_CONFIG["max_seq_length"]
)

print(f"✓ Instruction dataset created!")
print(f"  Total samples: {len(instruct_dataset):,}")
print(f"  Max sequence length: {FINETUNE_CONFIG['max_seq_length']:,}")

# Sample verification
sample = instruct_dataset[0]
print(f"\n  Sample 0 token count: {len(sample['input_ids'])}")

# Create data collator for fine-tuning
ft_data_collator = DataCollatorForLanguageModeling(
    tokenizer=ft_tokenizer,
    mlm=False,  # Causal LM
)
print("✓ DataCollator configured for fine-tuning")

---

# SECTION 12: INSTRUCTION FINE-TUNING

> Fine-tune the pre-trained model on instruction data

In [ ]:
# ============================================================================
# SECTION 12.1: FINE-TUNING TRAINING ARGUMENTS
# ============================================================================
"""
Configure Trainer for instruction fine-tuning.
Key differences from pre-training:
  - Lower learning rate (1e-5 vs 5e-5)
  - Fewer epochs (3 vs 10)
  - Warmup ratio instead of steps (adapts to dataset size)
"""

from transformers import TrainingArguments

ft_training_args = TrainingArguments(
    output_dir=FINETUNE_CONFIG["output_dir"],
    overwrite_output_dir=True,
    
    # Training duration
    num_train_epochs=FINETUNE_CONFIG["num_train_epochs"],
    
    # Batch size and accumulation
    per_device_train_batch_size=FINETUNE_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=FINETUNE_CONFIG["gradient_accumulation_steps"],
    
    # Optimizer settings (LOWER LR for fine-tuning)
    learning_rate=FINETUNE_CONFIG["learning_rate"],
    weight_decay=FINETUNE_CONFIG["weight_decay"],
    warmup_ratio=FINETUNE_CONFIG["warmup_ratio"],
    
    # Mixed precision
    fp16=FINETUNE_CONFIG["fp16"],
    
    # Logging
    logging_dir=FINETUNE_CONFIG["logging_dir"],
    logging_steps=FINETUNE_CONFIG["logging_steps"],
    
    # Checkpointing
    save_strategy=FINETUNE_CONFIG["save_strategy"],
    save_total_limit=FINETUNE_CONFIG["save_total_limit"],
    
    # Misc
    dataloader_num_workers=FINETUNE_CONFIG["dataloader_num_workers"],
    seed=FINETUNE_CONFIG["seed"],
    report_to=FINETUNE_CONFIG["report_to"],
)

print("✓ Fine-tuning TrainingArguments configured!")
print(f"  Output directory: {ft_training_args.output_dir}")
print(f"  Epochs: {ft_training_args.num_train_epochs}")
print(f"  Learning rate: {ft_training_args.learning_rate} (fine-tuning)")
print(f"  Warmup ratio: {ft_training_args.warmup_ratio}")
print(f"  Effective batch size: {ft_training_args.per_device_train_batch_size * ft_training_args.gradient_accumulation_steps}")

In [ ]:
# ============================================================================
# SECTION 12.2: INITIALIZE FINE-TUNING TRAINER
# ============================================================================
"""
Create Trainer instance for instruction fine-tuning.
Uses the pre-trained model from Phase 1.
"""

from transformers import Trainer

ft_trainer = Trainer(
    model=ft_model,
    args=ft_training_args,
    train_dataset=instruct_dataset,
    data_collator=ft_data_collator,
    tokenizer=ft_tokenizer,
)

print("✓ Fine-tuning Trainer initialized!")
print(f"  Model: {ft_model.__class__.__name__} (from Phase 1)")
print(f"  Dataset size: {len(instruct_dataset):,} instruction samples")

In [ ]:
# ============================================================================
# SECTION 12.3: EXECUTE FINE-TUNING
# ============================================================================
"""
Run the instruction fine-tuning training loop.
"""

print("=" * 60)
print("PHASE 2: STARTING INSTRUCTION FINE-TUNING")
print("=" * 60)
print(f"  Base model: Pre-trained GPT-2 Recipe Model")
print(f"  Dataset: {len(instruct_dataset):,} instruction samples")
print(f"  Epochs: {FINETUNE_CONFIG['num_train_epochs']}")
print(f"  Learning rate: {FINETUNE_CONFIG['learning_rate']}")
print(f"  Device: {device}")
print("=" * 60)

# Fine-tune the model
ft_train_result = ft_trainer.train()

# Print training summary
print("\n" + "=" * 60)
print("FINE-TUNING COMPLETE")
print("=" * 60)
print(f"  Total steps: {ft_train_result.global_step:,}")
print(f"  Training loss: {ft_train_result.training_loss:.4f}")
print("=" * 60)

In [ ]:
# ============================================================================
# SECTION 12.4: PLOT FINE-TUNING LOSS CURVE
# ============================================================================
"""
Visualize fine-tuning progress.
"""

# Extract loss history from trainer
ft_loss_history = [log["loss"] for log in ft_trainer.state.log_history if "loss" in log]
ft_steps = [log["step"] for log in ft_trainer.state.log_history if "loss" in log]

# Create loss curve plot
plt.figure(figsize=(12, 6))
plt.plot(ft_steps, ft_loss_history, 'g-', linewidth=2, alpha=0.8)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Phase 2: Instruction Fine-Tuning Loss Curve", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("finetuning_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Fine-tuning loss curve saved to: finetuning_loss_curve.png")

In [ ]:
# ============================================================================
# SECTION 12.5: SAVE FINE-TUNED MODEL
# ============================================================================
"""
Save the instruction-tuned model for deployment.
"""

INSTRUCT_MODEL_PATH = "gpt2_recipe_instruct_final"

# Save model
ft_model.save_pretrained(INSTRUCT_MODEL_PATH)
print(f"✓ Instruction-tuned model saved to: {INSTRUCT_MODEL_PATH}/")

# Save tokenizer
ft_tokenizer.save_pretrained(INSTRUCT_MODEL_PATH)
print(f"✓ Tokenizer saved to: {INSTRUCT_MODEL_PATH}/")

# List saved files
saved_files = os.listdir(INSTRUCT_MODEL_PATH)
print(f"\nSaved artifacts:")
for f in saved_files:
    size = os.path.getsize(os.path.join(INSTRUCT_MODEL_PATH, f)) / 1e6
    print(f"  {f}: {size:.2f} MB")

---

# SECTION 13: INTERACTIVE INFERENCE

> Chat with your instruction-tuned recipe model!

In [ ]:
# ============================================================================
# SECTION 13.1: INSTRUCTION-FOLLOWING GENERATION FUNCTION
# ============================================================================
"""
Generate recipes from user instructions using the fine-tuned model.
Formats the input in Alpaca-style and extracts the response.

IMPORTANT: Model is trained on STRUCTURED FORMAT with:
- **Title:** Recipe Name
- **Ingredients:** Bulleted list with -
- **Instructions:** Numbered steps with 1., 2., etc.
- **Serving Suggestion:** Optional serving guidance
"""

def validate_structured_output(response: str) -> dict:
    """
    Validate that the generated response follows structured format.
    
    Args:
        response: Generated recipe text
        
    Returns:
        dict with validation results and parsed sections
    """
    result = {
        "is_structured": False,
        "has_title": "**Title:**" in response,
        "has_ingredients": "**Ingredients:**" in response,
        "has_instructions": "**Instructions:**" in response,
        "has_serving": "**Serving Suggestion:**" in response,
        "has_bullets": any(f"\n- " in response or "- " in response for _ in [1]),
        "has_numbers": any(f"\n{i}." in response or f" {i}." in response for i in range(1, 10)),
        "sections": {}
    }
    
    # Fully structured if has all three required markers
    result["is_structured"] = (
        result["has_title"] and 
        result["has_ingredients"] and 
        result["has_instructions"]
    )
    
    # Try to extract sections
    try:
        if result["has_title"]:
            title_start = response.find("**Title:**") + len("**Title:**")
            title_end = response.find("\n", title_start)
            result["sections"]["title"] = response[title_start:title_end].strip() if title_end > title_start else ""
    except:
        pass
    
    return result


def generate_from_instruction(instruction: str, 
                               max_new_tokens: int = 300,
                               temperature: float = 0.8,
                               top_k: int = 50,
                               top_p: float = 0.92,
                               validate: bool = True) -> tuple:
    """
    Generate a recipe from a user instruction.
    
    Args:
        instruction: User's request (e.g., "Give me a recipe for chocolate cake")
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature
        top_k: Top-k sampling parameter
        top_p: Nucleus sampling parameter
        validate: Whether to validate structured output format
        
    Returns:
        tuple: (generated_text, validation_result) if validate=True
               else just generated_text
    """
    # Format as instruction prompt (without response - model will generate it)
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    
    # Tokenize
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(device)
    prompt_length = inputs["input_ids"].shape[1]
    
    # Generate
    ft_model.eval()
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            pad_token_id=ft_tokenizer.pad_token_id,
            eos_token_id=ft_tokenizer.eos_token_id,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
        )
    
    # Decode only the generated part (after prompt)
    generated_ids = outputs[0][prompt_length:]
    response = ft_tokenizer.decode(generated_ids, skip_special_tokens=True)
    response = response.strip()
    
    if validate:
        validation = validate_structured_output(response)
        return response, validation
    else:
        return response


def format_for_display(response: str) -> str:
    """
    Format structured response for nice display.
    Converts inline markers to proper formatting.
    """
    # Replace inline \n markers with actual newlines if present
    formatted = response.replace(" \\n ", "\n").replace("\\n", "\n")
    
    # Add emoji headers for sections
    formatted = formatted.replace("**Title:**", "🍽️ **Title:**")
    formatted = formatted.replace("**Ingredients:**", "\n📝 **Ingredients:**")
    formatted = formatted.replace("**Instructions:**", "\n👨‍🍳 **Instructions:**")
    formatted = formatted.replace("**Serving Suggestion:**", "\n🍴 **Serving Suggestion:**")
    
    return formatted


print("✓ Generation functions defined!")
print("\nFunctions available:")
print("  • generate_from_instruction(instruction) → (response, validation)")
print("  • validate_structured_output(response) → validation dict")
print("  • format_for_display(response) → formatted string")
print("\nExpected structured output format:")
print("  **Title:** Recipe Name")
print("  **Ingredients:**")
print("  - Ingredient 1")
print("  - Ingredient 2")
print("  **Instructions:**")
print("  1. Step one.")
print("  2. Step two.")
print("  **Serving Suggestion:** Serve hot.")

In [ ]:
# ============================================================================
# SECTION 13.2: EXAMPLE GENERATIONS WITH VALIDATION
# ============================================================================
"""
Demonstrate instruction-following with example prompts.
Validates that each response follows the structured format.
"""

print("=" * 70)
print("INSTRUCTION-TUNED MODEL EXAMPLES (WITH STRUCTURE VALIDATION)")
print("=" * 70)

# Example instructions
example_instructions = [
    "Give me a recipe for chocolate cake",
    "How do I make pasta carbonara?",
    "I want a quick breakfast recipe with eggs",
    "Create a vegetarian dinner recipe",
]

# Track validation stats
validation_stats = {
    "total": 0,
    "structured": 0,
    "has_title": 0,
    "has_ingredients": 0,
    "has_instructions": 0,
}

for instruction in example_instructions:
    print(f"\n{'─' * 70}")
    print(f"📝 INSTRUCTION: {instruction}")
    print('─' * 70)
    
    # Generate with validation
    response, validation = generate_from_instruction(instruction)
    
    # Update stats
    validation_stats["total"] += 1
    if validation["is_structured"]:
        validation_stats["structured"] += 1
    if validation["has_title"]:
        validation_stats["has_title"] += 1
    if validation["has_ingredients"]:
        validation_stats["has_ingredients"] += 1
    if validation["has_instructions"]:
        validation_stats["has_instructions"] += 1
    
    # Display formatted response
    print(f"\n🍳 RESPONSE:")
    print(format_for_display(response))
    
    # Display validation status
    print(f"\n📊 STRUCTURE VALIDATION:")
    status = "✅ PASS" if validation["is_structured"] else "⚠️ PARTIAL"
    print(f"   Overall: {status}")
    print(f"   • Title:        {'✓' if validation['has_title'] else '✗'}")
    print(f"   • Ingredients:  {'✓' if validation['has_ingredients'] else '✗'}")
    print(f"   • Instructions: {'✓' if validation['has_instructions'] else '✗'}")
    print(f"   • Bullets (-):  {'✓' if validation['has_bullets'] else '✗'}")
    print(f"   • Numbers (1.): {'✓' if validation['has_numbers'] else '✗'}")

# Summary statistics
print("\n" + "=" * 70)
print("📈 VALIDATION SUMMARY")
print("=" * 70)
pct = lambda k: 100 * validation_stats[k] / validation_stats["total"]
print(f"  Fully Structured: {validation_stats['structured']}/{validation_stats['total']} ({pct('structured'):.0f}%)")
print(f"  Has Title:        {validation_stats['has_title']}/{validation_stats['total']} ({pct('has_title'):.0f}%)")
print(f"  Has Ingredients:  {validation_stats['has_ingredients']}/{validation_stats['total']} ({pct('has_ingredients'):.0f}%)")
print(f"  Has Instructions: {validation_stats['has_instructions']}/{validation_stats['total']} ({pct('has_instructions'):.0f}%)")

# Quality gate
if pct('structured') >= 75:
    print(f"\n✅ QUALITY GATE PASSED: {pct('structured'):.0f}% structured output")
else:
    print(f"\n⚠️ QUALITY WARNING: Only {pct('structured'):.0f}% structured output")
    print("   Model may need more training on structured data.")
print("=" * 70)

In [ ]:
# ============================================================================
# SECTION 13.3: INTERACTIVE CHAT LOOP WITH VALIDATION
# ============================================================================
"""
Interactive loop for chatting with the recipe model.
Validates each response for structured format.
Type 'quit' or 'exit' to stop.
"""

print("=" * 70)
print("🍳 RECIPE ASSISTANT - INTERACTIVE MODE")
print("=" * 70)
print("Ask me for any recipe! Type 'quit' or 'exit' to stop.")
print("\nExpected output format:")
print("  **Title:** Recipe Name")
print("  **Ingredients:** Bulleted list")
print("  **Instructions:** Numbered steps")
print("=" * 70)
print()

# Track session stats
session_stats = {"total": 0, "structured": 0}

while True:
    try:
        # Get user input
        user_input = input("📝 Your request: ").strip()
        
        # Check for exit
        if user_input.lower() in ['quit', 'exit', 'q']:
            # Print session summary
            if session_stats["total"] > 0:
                pct = 100 * session_stats["structured"] / session_stats["total"]
                print(f"\n📊 Session Summary: {session_stats['structured']}/{session_stats['total']} ({pct:.0f}%) structured responses")
            print("\n👋 Goodbye! Happy cooking!")
            break
        
        if not user_input:
            print("   Please enter a recipe request.\n")
            continue
        
        # Generate response with validation
        print("\n🔄 Generating recipe...\n")
        response, validation = generate_from_instruction(user_input)
        
        # Update session stats
        session_stats["total"] += 1
        if validation["is_structured"]:
            session_stats["structured"] += 1
        
        # Display formatted response
        print("─" * 70)
        print(f"🍳 RECIPE:\n")
        print(format_for_display(response))
        print("─" * 70)
        
        # Show validation status
        status = "✅ Structured" if validation["is_structured"] else "⚠️ Partial structure"
        markers = []
        if validation["has_title"]: markers.append("Title")
        if validation["has_ingredients"]: markers.append("Ingredients")
        if validation["has_instructions"]: markers.append("Instructions")
        print(f"📊 Format: {status} | Found: {', '.join(markers) if markers else 'None'}")
        print()
        
    except KeyboardInterrupt:
        if session_stats["total"] > 0:
            pct = 100 * session_stats["structured"] / session_stats["total"]
            print(f"\n📊 Session Summary: {session_stats['structured']}/{session_stats['total']} ({pct:.0f}%) structured responses")
        print("\n\n👋 Goodbye! Happy cooking!")
        break
    except Exception as e:
        print(f"   Error: {str(e)}\n")

---

# SECTION 14: PHASE 2 EXPORT & SUMMARY

> Download fine-tuned model and view training summary

In [ ]:
# ============================================================================
# SECTION 14.1: DOWNLOAD FINE-TUNED MODEL (COLAB)
# ============================================================================
"""
Zip and download the instruction-tuned model for local storage.
"""

import shutil

# Create zip archive
instruct_zip_name = "gpt2_recipe_instruct_model"
shutil.make_archive(instruct_zip_name, 'zip', INSTRUCT_MODEL_PATH)
print(f"✓ Created archive: {instruct_zip_name}.zip")

# Download in Colab (uncomment when running in Colab)
# from google.colab import files
# files.download(f"{instruct_zip_name}.zip")

print("\nTo download in Colab, uncomment the files.download() line above.")

In [ ]:
# ============================================================================
# SECTION 14.2: COMPLETE PIPELINE SUMMARY
# ============================================================================
"""
Display summary of the complete two-phase training pipeline.
"""

print("=" * 70)
print("🎉 COMPLETE TRAINING PIPELINE SUMMARY")
print("=" * 70)

print("\n📊 PHASE 1: PRE-TRAINING (Domain Adaptation)")
print("─" * 70)
print(f"  Dataset: {len(recipes):,} raw recipes")
print(f"  Tokenizer: Custom BPE with {ft_tokenizer.vocab_size:,} vocabulary")
print(f"  Model: GPT-2 Mini (~{sum(p.numel() for p in ft_model.parameters()):,} parameters)")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  Final Loss: {train_result.training_loss:.4f}")
print(f"  Output: {PRETRAINED_MODEL_PATH}/")

print("\n📊 PHASE 2: INSTRUCTION FINE-TUNING (Alignment)")
print("─" * 70)
print(f"  Dataset: {len(instruction_data):,} instruction-response pairs")
print(f"  Format: Alpaca-style (### Instruction / ### Response)")
print(f"  Epochs: {FINETUNE_CONFIG['num_train_epochs']}")
print(f"  Learning Rate: {FINETUNE_CONFIG['learning_rate']} (10x lower)")
print(f"  Final Loss: {ft_train_result.training_loss:.4f}")
print(f"  Output: {INSTRUCT_MODEL_PATH}/")

print("\n🚀 USAGE")
print("─" * 70)
print("To load the instruction-tuned model later:")
print(f"  model = GPT2LMHeadModel.from_pretrained('{INSTRUCT_MODEL_PATH}')")
print(f"  tokenizer = GPT2TokenizerFast.from_pretrained('{INSTRUCT_MODEL_PATH}')")
print()
print("To generate recipes:")
print("  prompt = '### Instruction:\\nGive me a recipe for pasta\\n\\n### Response:\\n'")
print("  inputs = tokenizer(prompt, return_tensors='pt')")
print("  outputs = model.generate(**inputs, max_new_tokens=300)")

print("\n" + "=" * 70)
print("✅ Pipeline complete! Your recipe assistant is ready.")
print("=" * 70)